# A1.5 · Multi-agent topology

**Function A — Security Architecture & Platform → The Security Architect**  ·  *Security of AI*

---

**Risk.** Fan-out concentrates authority somewhere nobody drew.

**Control.** Planner/executor/critic splits with delegation-depth limits.

**This lab.** Find where authority actually concentrates in a multi-agent topology.

| | |
|---|---|
| Open-source tooling | kagent |
| Open-weight models | GLM-4.6, Llama 3.3 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("A1.5"))

Multi-agent topologies fail in the seams. Delegation depth is the variable nobody bounds, and each hop is a place authority can widen.

In [ ]:
from cybercommons import identity

reg = identity.Registry()
root = reg.record(identity.mint("alice"))

# a three-hop chain, each hop narrowing
hop1 = reg.record(identity.exchange(root, "reviewer-agent", {"repo:read"}))
hop2 = reg.record(identity.exchange(root, "patch-agent",    {"repo:read", "repo:write"}))
hop3 = reg.record(identity.exchange(hop2, "deploy-agent",   {"repo:read"}))

for t in (root, hop1, hop2, hop3):
    print(t.describe())

print("\ndepth of the deepest chain:", max(len(t.chain()) for t in reg.issued))

Now the seam: what happens to the topology when one node is revoked?

In [ ]:
affected = reg.revoke("patch-agent")
print(f"revoking patch-agent invalidates {affected} token(s)\n")
for t in (hop1, hop2, hop3):
    ok, why = reg.valid(t)
    print(f"  {' → '.join(t.chain()):48s} valid={str(ok):5s} {why}")

`deploy-agent` was never revoked, but it derived its authority through `patch-agent`, so it dies too — correctly. A topology where revoking a middle node leaves its descendants running is a topology with no containment story.

### Expect

Three tokens carry increasingly narrow scopes and a readable `alice → … → agent` chain. Revoking `patch-agent` invalidates both it and the `deploy-agent` token derived from it, while `reviewer-agent` keeps working.

### Your turn

Add a fourth hop and a policy that refuses any exchange producing a chain longer than three actors. Where should that check live — at the issuer, or at the resource server?

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/A1.5.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*